# 02. 프루닝 + 양자화 + 평가 (L4, 약 3시간 / 약 5~6유닛)
변형마다 **프루닝(CPU) → GGUF 변환 → 객관식·퍼플렉시티 평가 → 속도·전력 측정 → 파일 삭제** 순서로 진행합니다.
결과 JSON은 변형이 끝날 때마다 드라이브에 저장되므로, 세션이 끊겨도 다시 실행하면 끝난 변형은 건너뜁니다.

| 단계 | 변형 |
|---|---|
| 기준 | `original` (전문가 32개) |
| 프루닝 | config의 `pruning.variants` (한국어/영어 보정 × 24/16개, 빈도 기준 비교) |
| 양자화 | `ko-reap-k16`을 MXFP4_MOE / Q8_0 / Q4_0 로 |

In [ ]:
# 저장소 받기 + 패키지 설치
BRANCH = "ccr-a463bea6-bpbyrh"   # main에 합친 뒤에는 "main"
REPO = "github.com/b1bibig/Cfinetune"
import os
try:  # 저장소가 비공개면 코랩 왼쪽 열쇠(보안 비밀)에 GH_TOKEN 을 넣어 두세요
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{_tok}@{REPO}"
except Exception:
    REPO_URL = f"https://{REPO}"
if not os.path.exists("/content/Cfinetune"):
    !git clone -q -b {BRANCH} {REPO_URL} /content/Cfinetune
%cd /content/Cfinetune
!git pull -q
%pip install -q -r requirements-colab.txt

In [ ]:
# 구글 드라이브: 결과(results/)를 세션이 끊겨도 남도록 저장
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, sys
DRIVE = "/content/drive/MyDrive/cfinetune"
os.makedirs(DRIVE, exist_ok=True)

def pull_results():
    if os.path.exists(f"{DRIVE}/results"):
        shutil.copytree(f"{DRIVE}/results", "results", dirs_exist_ok=True)

def push_results():
    shutil.copytree("results", f"{DRIVE}/results", dirs_exist_ok=True)
    print("드라이브에 결과 저장함")

pull_results()
sys.path.insert(0, "scripts"); sys.path.insert(0, "src")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU 메모리 {mem:.0f}GB")
assert mem >= 20, "런타임 > 런타임 유형 변경에서 L4 를 고르세요"

In [ ]:
# llama.cpp 빌드 (처음 약 15분, 이후에는 드라이브 캐시에서 복사)
!bash scripts/build_llama_cpp.sh /content/llama.cpp {DRIVE}/cache

In [ ]:
from _common import resolve_model_dir, load_config
cfg = load_config()
SRC = resolve_model_dir(cfg["model"]["id"])   # 원본 MXFP4 (약 13GB)
GGUF, PRUNED = "/content/gguf", "/content/models/pruned"
os.makedirs(GGUF, exist_ok=True)

def done(name):
    return os.path.exists(f"results/eval/{name}.json") and os.path.exists(f"results/bench/{name}.json")

def evaluate(name, gguf):
    !python scripts/05_eval.py --gguf {gguf} --name {name}
    !python scripts/06_bench.py --gguf {gguf} --name {name}
    push_results()

## 기준: 원본

In [ ]:
if not done("original"):
    !python scripts/03_make_gguf.py --hf-dir {SRC} --out {GGUF}/original.gguf
    evaluate("original", f"{GGUF}/original.gguf")
    os.remove(f"{GGUF}/original.gguf")

## 프루닝 변형

In [ ]:
for v in cfg["pruning"]["variants"]:
    name = v["name"]
    if done(name):
        print("건너뜀:", name); continue
    !python scripts/02_prune.py --variant {name} --src {SRC} --out-dir {PRUNED}
    !python scripts/03_make_gguf.py --hf-dir {PRUNED}/{name} --out {GGUF}/{name}.gguf
    evaluate(name, f"{GGUF}/{name}.gguf")
    os.remove(f"{GGUF}/{name}.gguf")
    if name != cfg["quantization"]["base_variant"]:
        shutil.rmtree(f"{PRUNED}/{name}")

## 양자화 비교

In [ ]:
base = cfg["quantization"]["base_variant"]
todo = [q for q in cfg["quantization"]["types"] if not done(f"{base}-{q}")]
if todo:
    if not os.path.exists(f"{PRUNED}/{base}"):
        !python scripts/02_prune.py --variant {base} --src {SRC} --out-dir {PRUNED}
    QS = " ".join(todo)
    !python scripts/03_make_gguf.py --hf-dir {PRUNED}/{base} --out {GGUF}/{base}.gguf --quant {QS}
    for q in todo:
        evaluate(f"{base}-{q}", f"{GGUF}/{base}-{q}.gguf")
        os.remove(f"{GGUF}/{base}-{q}.gguf")

## 중간 결과 표

In [ ]:
!python scripts/07_econ.py --measured-gpu "L4 (Colab)"
push_results()